# V5b' — loop reactance and R_rad diagnostic (addendum step 3, 5/11)

Square copper/PEC loop of the CST sheet V5 (side 20 mm on the centre line, **w = 2 mm**), cosine edge grading.
Question: the MoM loop looks too inductive (X/ω ≈ 66 nH at 0.9 GHz against ≈ 48 nH in CST and ≈ 47 nH from Grover's formula). Is it
Z itself, or the delta-gap port?

Light computations only (single frequencies, no sweep):

1. **Convergence of L = X/ω** at 0.889 GHz (untuned loop, one port) vs n_width and n_side, plus the same at 0.2 GHz to separate
   a static inductance L_s from a shunt capacitance C across the port: L(ω) = L_s / (1 − ω² L_s C).
2. **Port model**: delta cut vs Ding's enhanced gap source (`gap_source.multiple_gap_sources`, 0.5 mm window) on a mesh refined at the feed.
3. **Magnetic energy**: L = 4 W_m / |i|² from the vector-potential part of Z only (no port in the definition).
4. **Two R_rad** at resonance (PEC, capacitor re-tuned to 0.889 GHz): Re Z_in vs 2 P_rad / |i|² with P_rad integrated on the sphere.
5. **|J| across the strip width** at the middle of a side, n_width = 4 and 8.

No existing module is modified: everything below only calls them.

In [1]:
import sys, time, json, importlib.util
from pathlib import Path
import numpy as np

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from solver.optim import ReducedMultiport, incidence_matrix, port_from_cut
from solver.optim.farfield import RadiationOperator
from solver.optim.geometries import square_loop
from solver.optim.geometries.planar import tensor_grid_mesh
from solver.optim.threads import limit_blas_threads, threads_for_size
from solver.rwg.rwg3 import calculate_z_matrice_sweep
from solver.utils.impmet import impedance_matrix_z
from solver.utils.gap_source import multiple_gap_sources

_spec = importlib.util.spec_from_file_location('v5_mom_runs', NB_DIR / 'v5_mom_runs.py')
v5 = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(v5)

EPS0, MU0 = 8.854e-12, 1.257e-6                 # as rwg3.calculate_z_matrice
C_LIGHT = 1 / np.sqrt(EPS0 * MU0)
F0, F_LOW = 0.889e9, 0.2e9
W0, W_LOW = 2 * np.pi * F0, 2 * np.pi * F_LOW
SIDE, W = 0.020, 0.002
A_IN, A_OUT = SIDE / 2 - W / 2, SIDE / 2 + W / 2

# References
L_GROVER = 2 * MU0 * SIDE / np.pi * (np.log(SIDE / (W / 4)) - 0.774)   # square loop, flat strip r_eq = w/4
L_CST_UNTUNED = 272.8 / (2 * np.pi * 0.9e9)                            # CST V5b PEC w = 2: X(0.9 GHz) / w
LAM = C_LIGHT / F0
R_SMALL_LOOP = 31171 * (SIDE ** 2 / LAM ** 2) ** 2                     # uniform-current small loop
FIG_DIR = Path(r'D:\PhD_Fahdilou_Leat\PUBLICATIONS\7-EurAAP Small Antenna Super-Gain Challenge\fiches_cst\resultats_V5\figures')
print(f'L Grover = {L_GROVER*1e9:.1f} nH | CST untuned X/w (0.9 GHz) = {L_CST_UNTUNED*1e9:.1f} nH | '
      f'R_rad small loop (uniform current) = {R_SMALL_LOOP:.3f} Ohm at {F0/1e9} GHz')

L Grover = 46.7 nH | CST untuned X/w (0.9 GHz) = 48.2 nH | R_rad small loop (uniform current) = 0.386 Ohm at 0.889 GHz


In [2]:
from solver.optim.reduced_model import terminated_input_impedance


def assemble(mesh, freqs=(F_LOW, F0)):
    """Z at the given frequencies (one call, GPU when available)."""
    t = time.perf_counter()
    *_, Z = calculate_z_matrice_sweep(mesh.points, mesh.triangles, mesh.edges, np.array(freqs))
    print(f'  N = {mesh.n_edges}: Z at {len(freqs)} frequencies in {time.perf_counter() - t:.1f} s')
    return dict(zip(freqs, Z))


def shunt_model(L_low, L_hi):
    """L(w) = L_s / (1 - w^2 L_s C)  <=>  1/L(w) = 1/L_s - w^2 C : two frequencies give L_s and C."""
    C = (1 / L_low - 1 / L_hi) / (W0 ** 2 - W_LOW ** 2)
    return 1 / (1 / L_low + W_LOW ** 2 * C), C


def tune(zp, mesh_label):
    """Capacitive load reactance making Im Zin = 0; falls back to a scan of X < 0 (with a printout) if the
    closed form of v5_mom_runs.tuning_reactance finds no capacitive root."""
    X_t = v5.tuning_reactance(zp, x_guess=-1 / (W0 * 0.5e-12))
    if X_t is not None:
        return X_t
    p = zp[0, 1] * zp[1, 0]
    a, r, x22 = zp[0, 0].imag, zp[1, 1].real, zp[1, 1].imag
    print(f'  {mesh_label}: no capacitive root in closed form. Z11 = {zp[0,0]:.4f}, Z22 = {zp[1,1]:.4f}, '
          f'Z12 Z21 = {p:.4f}, roots X = {np.roots([a, p.real, a * r * r - p.imag * r]) - x22}')
    Xs = -np.logspace(-1, 4, 50001)
    zs = terminated_input_impedance(np.broadcast_to(zp, (Xs.size, 2, 2)), (1j * Xs)[:, None])
    k = np.flatnonzero(np.sign(zs.imag[:-1]) != np.sign(zs.imag[1:]))
    print(f'  scan X in [-1e4, -0.1] Ohm: Im Zin from {zs.imag.min():.4g} to {zs.imag.max():.4g}; sign changes at X = {Xs[k]}')
    if k.size == 0:
        return None
    k = k[np.argmin(np.abs(Xs[k] + 1 / (W0 * 0.5e-12)))]
    return Xs[k] - zs.imag[k] * (Xs[k + 1] - Xs[k]) / (zs.imag[k + 1] - zs.imag[k])


def analyse(mesh, Zf, A1, A2, label=''):
    """
    A1 (N,1): feed only (untuned loop). A2 (N,2): feed + load cut (capacitor).
    The feed column may be any real excitation vector V (delta cut a_p, or Ding's V):
    Z_in = v / (V^T I) is then the power-consistent input impedance |v|^2 / (V^H I).
    """
    zin = {f: 1 / ReducedMultiport.from_matrices(Z, A1, f).Y[0, 0] for f, Z in Zf.items()}
    L_low, L_hi = zin[F_LOW].imag / W_LOW, zin[F0].imag / W0
    L_s, C_sh = shunt_model(L_low, L_hi)
    out = dict(N=mesh.n_edges, Zin_untuned=zin[F0], Zin_untuned_low=zin[F_LOW], L_low_nH=L_low * 1e9,
               L_f0_nH=L_hi * 1e9, L_s_nH=L_s * 1e9, C_shunt_pF=C_sh * 1e12)
    print(f'  {label}: L(0.2 GHz) = {L_low*1e9:.2f} nH, L(0.889 GHz) = {L_hi*1e9:.2f} nH, Re Zin(0.889) = {zin[F0].real:.4f} Ohm')
    m2 = ReducedMultiport.from_matrices(Zf[F0], A2, F0)
    X_t = tune(np.linalg.inv(m2.Y), label)
    if X_t is None:
        return dict(out, C_tuned_pF=np.nan, R_in=np.nan, X_in=np.nan, R_rad_M=np.nan, R_rad_sphere=np.nan, I_tuned=None)
    sol = m2.solve([1.0, 0.0], [1.0, 1j * X_t])                     # 1 Ohm generator, as CST
    I = m2.edge_currents(sol.v)
    p_sph = RadiationOperator(mesh.points, mesh.triangles, mesh.edges, F0).radiated_power_sphere(I)
    i2 = abs(sol.i[0]) ** 2
    return dict(out, C_tuned_pF=-1 / (W0 * X_t) * 1e12, R_in=sol.z_in[0].real, X_in=sol.z_in[0].imag,
                R_rad_M=2 * sol.p_rad / i2, R_rad_sphere=2 * p_sph / i2, I_tuned=I)


def cut_ports(mesh):
    return (incidence_matrix([mesh.port('feed')], mesh.edges),
            incidence_matrix([mesh.port('feed'), mesh.port('load')], mesh.edges))


def feed_cell_length(mesh):
    x = np.abs(mesh.xs[mesh.xs != 0])
    return x.min()

## 1. Convergence of L = X/ω (delta-cut port), and 4. the two R_rad

In [3]:
CASES = [('ns16_nw2', 16, 2), ('ns16_nw4', 16, 4), ('ns16_nw8', 16, 8), ('ns8_nw4', 8, 4), ('ns32_nw4', 32, 4)]
meshes, Zs, res = {}, {}, {}
for name, ns, nw in CASES:
    meshes[name] = square_loop(SIDE, W, ns, nw, 'cosine')
    Zs[name] = assemble(meshes[name])
    with limit_blas_threads(threads_for_size(meshes[name].n_edges)):
        res[name] = analyse(meshes[name], Zs[name], *cut_ports(meshes[name]), label=name)

print(f"\n{'mesh':10s} {'N':>5s} {'dx feed':>8s} {'L(0.2 GHz)':>11s} {'L(0.889)':>9s} {'L_s':>7s} {'C_shunt':>8s} {'Re Zin untuned':>15s}")
for name, *_ in CASES:
    r = res[name]
    print(f"{name:10s} {r['N']:5d} {feed_cell_length(meshes[name])*1e3:6.3f}mm {r['L_low_nH']:9.2f}nH {r['L_f0_nH']:7.2f}nH "
          f"{r['L_s_nH']:5.2f}nH {r['C_shunt_pF']:6.4f}pF {r['Zin_untuned'].real:13.4f} Ohm")
print(f"references: Grover {L_GROVER*1e9:.1f} nH, CST X/w {L_CST_UNTUNED*1e9:.1f} nH")

[calculate_z_matrice_sweep] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (360x360 edges, 2 frequencies).


  N = 360: Z at 2 frequencies in 4.8 s
  ns16_nw2: L(0.2 GHz) = 83.85 nH, L(0.889 GHz) = 67.18 nH, Re Zin(0.889) = 0.8578 Ohm
[calculate_z_matrice_sweep] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (880x880 edges, 2 frequencies).


  N = 880: Z at 2 frequencies in 14.4 s


  ns16_nw4: L(0.2 GHz) = 83.20 nH, L(0.889 GHz) = 65.38 nH, Re Zin(0.889) = 0.8466 Ohm


[calculate_z_matrice_sweep] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (2208x2208 edges, 2 frequencies).


  N = 2208: Z at 2 frequencies in 62.5 s


... (2 outputs left out)

  N = 528: Z at 2 frequencies in 9.4 s
  ns8_nw4: L(0.2 GHz) = 34.96 nH, L(0.889 GHz) = -50.22 nH, Re Zin(0.889) = 0.5332 Ohm
  ns8_nw4: no capacitive root in closed form. Z11 = 1.2027-433.9652j, Z22 = 0.9206-712.9238j, Z12 Z21 = -109404.2393-618.5867j, roots X = [460.81826455 712.92567568]
  scan X in [-1e4, -0.1] Ohm: Im Zin from -423.8 to -280.5; sign changes at X = []
[calculate_z_matrice_sweep] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (1584x1584 edges, 2 frequencies).


  N = 1584: Z at 2 frequencies in 34.8 s


  ns32_nw4: L(0.2 GHz) = 55.56 nH, L(0.889 GHz) = 70.56 nH, Re Zin(0.889) = 0.9132 Ohm



mesh           N  dx feed  L(0.2 GHz)  L(0.889)     L_s  C_shunt  Re Zin untuned
ns16_nw2     360  1.125mm     83.85nH   67.18nH 84.97nH -0.0999pF        0.8578 Ohm
ns16_nw4     880  1.125mm     83.20nH   65.38nH 84.42nH -0.1106pF        0.8466 Ohm
ns16_nw8    2208  1.125mm  -1064.00nH   45.15nH -460.70nH -0.7794pF        0.6652 Ohm
ns8_nw4      528  2.250mm     34.96nH  -50.22nH 32.06nH 1.6378pF        0.5332 Ohm
ns32_nw4    1584  0.563mm     55.56nH   70.56nH 54.94nH 0.1292pF        0.9132 Ohm
references: Grover 46.7 nH, CST X/w 48.2 nH


In [4]:
print(f"{'mesh':10s} {'C_tuned':>8s} {'R_in (Re Zin)':>14s} {'2Prad/|i|^2 (M_rad)':>20s} {'2Prad/|i|^2 (sphere)':>21s} "
      f"{'R_in (L_f0/L_s)^2':>18s} {'Re Zin untuned':>15s}")
for name, *_ in CASES:
    r = res[name]
    print(f"{name:10s} {r['C_tuned_pF']:6.4f}pF {r['R_in']:12.4f} Ohm {r['R_rad_M']:16.4f} Ohm {r['R_rad_sphere']:17.4f} Ohm "
          f"{r['R_in'] * (r['L_f0_nH'] / r['L_s_nH'])**2:14.4f} Ohm {r['Zin_untuned'].real:11.4f} Ohm")
print(f"small-loop formula (uniform current): {R_SMALL_LOOP:.4f} Ohm")

mesh        C_tuned  R_in (Re Zin)  2Prad/|i|^2 (M_rad)  2Prad/|i|^2 (sphere)  R_in (L_f0/L_s)^2  Re Zin untuned
ns16_nw2   0.4769pF       0.4105 Ohm           0.4105 Ohm            0.4105 Ohm         0.2566 Ohm      0.8578 Ohm
ns16_nw4   0.4915pF       0.4042 Ohm           0.4042 Ohm            0.4042 Ohm         0.2424 Ohm      0.8466 Ohm
ns16_nw8   0.7243pF       0.3758 Ohm           0.3758 Ohm            0.3758 Ohm         0.0036 Ohm      0.6652 Ohm
ns8_nw4       nanpF          nan Ohm              nan Ohm               nan Ohm            nan Ohm      0.5332 Ohm
ns32_nw4   0.4531pF       0.4111 Ohm           0.4111 Ohm            0.4111 Ohm         0.6782 Ohm      0.9132 Ohm
small-loop formula (uniform current): 0.3859 Ohm


## 3. Magnetic energy: L = 4 W_m / |i|², vector-potential part of Z only

`impmet.impedance_matrix_z` does not return the A and Φ terms separately, but it takes k, ω, μ and ε as **independent** arguments
(`pref_A = jωμ/4π`, `pref_Phi = 1/(jωε4π)`, Green's function from k only). With ε → 10³⁰⁰ the Φ term vanishes and the same call
returns Z_A alone, with the right k. (The GPU path derives k from μ and ε, so this only works on the CPU per-pair assembler: done on the
two smallest meshes.)

With peak phasors, reactive power ½ X|i|² = ½ Iᴴ Im(Z) I = ½ Iᴴ Im(Z_A) I + ½ Iᴴ Im(Z_Φ) I = 2ω (W_m − W_e), hence
L_m = 4 W_m / |i|² = Iᴴ Im(Z_A) I / (ω |i|²).

In [5]:
wm = {}
k0 = W0 / C_LIGHT
EPS_HUGE = 1e250          # pref_Phi = 1/(j w eps 4 pi) -> ~1e-261; 1e300 overflows w*eps (inf -> NaN)
for name in ('ns16_nw2', 'ns8_nw4'):
    mesh, Z0 = meshes[name], Zs[name][F0]
    t = time.perf_counter()
    Z_A = impedance_matrix_z(mesh.edges, mesh.triangles, mesh.points, 1j * k0, W0, MU0, EPS_HUGE)
    t = time.perf_counter() - t
    assert np.all(np.isfinite(Z_A))
    A1, A2 = cut_ports(mesh)
    m1 = ReducedMultiport.from_matrices(Z0, A1, F0)
    I = m1.edge_currents([1.0])
    i_port, i_opp = A1[:, 0] @ I, A2[:, 1] @ I
    qA = float(np.real(I.conj() @ Z_A.imag @ I))
    q = float(np.real(I.conj() @ Z0.imag @ I))
    X = (1 / m1.Y[0, 0]).imag
    wm[name] = dict(L_X_nH=X / W0 * 1e9, L_check_nH=q / (W0 * abs(i_port)**2) * 1e9,
                    L_m_port_nH=qA / (W0 * abs(i_port)**2) * 1e9, L_m_opp_nH=qA / (W0 * abs(i_opp)**2) * 1e9,
                    i_opp_over_i=abs(i_opp) / abs(i_port), t_cpu_s=t)
    print(f"{name}: Z_A on CPU in {t:.0f} s | X/w = {wm[name]['L_X_nH']:.2f} nH (check I^H Im Z I: {wm[name]['L_check_nH']:.2f}) | "
          f"4W_m/|i_port|^2 = {wm[name]['L_m_port_nH']:.2f} nH | |i_opposite/i_port| = {wm[name]['i_opp_over_i']:.4f} | "
          f"4W_m/|i_opposite|^2 = {wm[name]['L_m_opp_nH']:.2f} nH")

ns16_nw2: Z_A on CPU in 16 s | X/w = 67.18 nH (check I^H Im Z I: 67.18) | 4W_m/|i_port|^2 = 86.70 nH | |i_opposite/i_port| = 1.4458 | 4W_m/|i_opposite|^2 = 41.47 nH


ns8_nw4: Z_A on CPU in 31 s | X/w = -50.22 nH (check I^H Im Z I: -50.22) | 4W_m/|i_port|^2 = 59407.40 nH | |i_opposite/i_port| = 0.4640 | 4W_m/|i_opposite|^2 = 275985.63 nH


## 2. Port model: delta cut vs Ding's enhanced gap (0.5 mm window)

Finest density (n_side = 16, n_width = 8) with extra x-lines at ±0.125, ±0.25, ±0.375, ±0.5, ±0.75 mm so that triangle centroids fall
inside a 0.5 mm window. `multiple_gap_sources` tests a **cube** of side `gap_width` around each feed point, so a 2 mm wide strip is
covered by 4 feed points across the width (each with the full voltage: the field is v/W over every slice); a check below makes sure no
triangle belongs to two windows. The load (capacitor) stays a delta cut on the opposite side.

In [6]:
GAP = 0.5e-3
base = square_loop(SIDE, W, 16, 8, 'cosine')
extra = np.array([0.125, 0.25, 0.375, 0.5, 0.75]) * 1e-3
xs = np.unique(np.concatenate([base.xs, extra, -extra]))
ys = base.ys
XC, YC = np.meshgrid(0.5 * (xs[:-1] + xs[1:]), 0.5 * (ys[:-1] + ys[1:]), indexing='ij')
keep = (np.abs(XC) > A_IN) | (np.abs(YC) > A_IN)
m6 = tensor_grid_mesh(xs, ys, keep, cuts=base.cuts, params=dict(base.params, feed_refinement_mm=list(extra * 1e3)))
meshes['ns16_nw8_ref'] = m6
Zs['ns16_nw8_ref'] = Z6 = assemble(m6)

# (a) delta cut on the refined mesh
with limit_blas_threads(threads_for_size(m6.n_edges)):
    res['ns16_nw8_ref'] = analyse(m6, Z6, *cut_ports(m6), label='delta, refined')

# (b) Ding enhanced gap source, full width tiled by 4 windows
feed_pts = np.array([[0.0, -A_OUT + GAP / 2 + GAP * k, 0.0] for k in range(int(round(W / GAP)))])
V, groups = multiple_gap_sources(m6.triangles, m6.edges, m6.vecteurs_rho, 1.0, feed_pts, 'x', GAP)
cent = m6.triangles.triangles_center
in_box = np.array([np.all(np.abs(cent - p[:, None]) <= GAP / 2, axis=0) for p in feed_pts])
assert in_box.sum(axis=0).max() <= 1, 'a triangle centroid lies in two windows'
assert np.allclose(V.imag, 0)
print(f'Ding source: {in_box.any(axis=0).sum()} triangles in the 0.5 mm x 2 mm window, {np.count_nonzero(V)} edges excited')
A1_d = V.real[:, None]
A2_d = np.column_stack([V.real, incidence_matrix([m6.port('load')], m6.edges)[:, 0]])
with limit_blas_threads(threads_for_size(m6.n_edges)):
    res['ns16_nw8_ref_ding'] = analyse(m6, Z6, A1_d, A2_d, label='Ding, refined')

print(f"\n{'port':32s} {'N':>5s} {'L(0.2 GHz)':>11s} {'L(0.889)':>9s} {'L_s':>8s} {'C_shunt':>9s} {'C_tuned':>9s} {'R_in tuned':>11s} {'R_rad sphere':>13s}")
for name, label in (('ns16_nw8', 'delta, dx_feed = 1.125 mm'), ('ns16_nw8_ref', 'delta, dx_feed = 0.125 mm'),
                    ('ns16_nw8_ref_ding', 'Ding 0.5 mm, same refined mesh')):
    r = res[name]
    print(f"{label:32s} {r['N']:5d} {r['L_low_nH']:9.2f}nH {r['L_f0_nH']:7.2f}nH {r['L_s_nH']:6.2f}nH {r['C_shunt_pF']:7.4f}pF "
          f"{r['C_tuned_pF']:7.4f}pF {r['R_in']:9.4f}Ohm {r['R_rad_sphere']:10.4f}Ohm")

[calculate_z_matrice_sweep] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (2668x2668 edges, 2 frequencies).


  N = 2668: Z at 2 frequencies in 81.3 s


  delta, refined: L(0.2 GHz) = -983.64 nH, L(0.889 GHz) = 48.16 nH, Re Zin(0.889) = 0.7197 Ohm


Ding source: 64 triangles in the 0.5 mm x 2 mm window, 100 edges excited


  Ding, refined: L(0.2 GHz) = -1037.24 nH, L(0.889 GHz) = 45.87 nH, Re Zin(0.889) = 0.6528 Ohm



port                                 N  L(0.2 GHz)  L(0.889)      L_s   C_shunt   C_tuned  R_in tuned  R_rad sphere
delta, dx_feed = 1.125 mm         2208  -1064.00nH   45.15nH -460.70nH -0.7794pF  0.7243pF    0.3758Ohm     0.3758Ohm
delta, dx_feed = 0.125 mm         2668   -983.64nH   48.16nH -459.20nH -0.7353pF  0.6797pF    0.3759Ohm     0.3759Ohm
Ding 0.5 mm, same refined mesh    2668  -1037.24nH   45.87nH -459.18nH -0.7686pF  0.6797pF    0.3757Ohm     0.3757Ohm


## 5. |J| across the strip width at the middle of a side (tuned PEC loop, 0.889 GHz)

Left side (x from −11 to −9 mm), row of cells just above y = 0. J is evaluated at triangle centroids,
J(c_t) = Σ_n ± I_n l_n/(2A_t) (c_t − v_n), and normalised by the current crossing the side at y = 0. Reference: the isolated-strip
(Maxwell) distribution 1/(π √((w/2)² − u²)).

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def j_at_centroids(mesh, I, tri_idx):
    tri, pts = mesh.triangles.triangles[:3], mesh.points.points
    plus, minus = mesh.triangles.triangles_plus, mesh.triangles.triangles_minus
    L, A, C = mesh.edges.edges_length, mesh.triangles.triangles_area, mesh.triangles.triangles_center
    fp, sp = mesh.edges.first_points, mesh.edges.second_points
    pos = {int(t): k for k, t in enumerate(tri_idx)}
    J = np.zeros((len(tri_idx), 3), dtype=complex)
    for n in range(mesh.n_edges):
        for t, s in ((int(plus[n]), 1.0), (int(minus[n]), -1.0)):
            if t in pos:
                free = [v for v in tri[:, t] if v not in (fp[n], sp[n])][0]
                J[pos[t]] += s * I[n] * L[n] / (2 * A[t]) * (C[:, t] - pts[:, free])
    return J


def save_png(fig, path):
    """Save; if the file is locked (open in a viewer), save under a time-stamped name instead."""
    try:
        fig.savefig(path, dpi=150)
    except OSError as exc:
        path = path.with_name(f"{path.stem}_{time.strftime('%H%M%S')}{path.suffix}")
        print(f'  ({type(exc).__name__}: target locked, saving as {path.name})')
        fig.savefig(path, dpi=150)
    print('wrote', path)
    return path


INK, SURFACE, GRID, AXIS, MUTED = '#0b0b0b', '#fcfcfb', '#e1e0d9', '#c3c2b7', '#898781'
DENSITY = {'ns16_nw4': '#3987e5', 'ns16_nw8': '#184f95'}
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'axes.edgecolor': AXIS, 'axes.grid': True, 'grid.color': GRID, 'axes.spines.top': False,
                     'axes.spines.right': False, 'legend.frameon': False, 'font.size': 9})
fig, ax = plt.subplots(figsize=(7, 4))
jprof = {}
for name in ('ns16_nw4', 'ns16_nw8'):
    mesh, I = meshes[name], res[name]['I_tuned']
    y1 = mesh.ys[mesh.ys > 1e-12].min()
    c = mesh.triangles.triangles_center
    sel = np.flatnonzero((c[0] > -A_OUT) & (c[0] < -A_IN) & (c[1] > 0) & (c[1] < y1))
    J = j_at_centroids(mesh, I, sel)
    side = port_from_cut(mesh.points, mesh.triangles, mesh.edges, [-A_OUT, 0, 0], [-A_IN, 0, 0], [0, -1, 0], 'side')
    i_side = side.incidence_vector(mesh.edges) @ I
    u = c[0, sel] + SIDE / 2                          # 0 on the centre line, -w/2 outer edge, +w/2 inner edge
    order = np.argsort(u)
    jn = np.linalg.norm(J, axis=1) / abs(i_side)
    jprof[name] = dict(u_mm=(u[order] * 1e3).tolist(), J_over_i_per_mm=(jn[order] / 1e3).tolist(),
                       i_side_over_i_port=float(abs(i_side) / abs(res[name]['I_tuned'] @ cut_ports(mesh)[0][:, 0])))
    ax.plot(u[order] * 1e3, jn[order] / 1e3, marker='o', ms=4, lw=1.6, color=DENSITY[name],
            label=f"MoM n_width = {name[-1]} (N = {mesh.n_edges})")
uu = np.linspace(-0.995, 0.995, 400) * W / 2
ax.plot(uu * 1e3, 1 / (np.pi * np.sqrt((W / 2) ** 2 - uu ** 2)) / 1e3, color=INK, lw=1.2, ls='--',
        label='isolated strip 1/(pi sqrt((w/2)^2 - u^2))')
ax.set_xlabel('position across the strip u (mm): -1 outer edge, +1 inner edge')
ax.set_ylabel('|J| / |i_side| (1/mm)')
ax.set_ylim(0, 2.0)
ax.set_title("V5b' tuned PEC loop, w = 2 mm, 0.889 GHz: current across the width (middle of the left side)", fontsize=9)
fig.legend(*ax.get_legend_handles_labels(), loc='lower center', ncol=2)
fig.tight_layout(rect=(0, 0.12, 1, 1))
FIG_DIR.mkdir(parents=True, exist_ok=True)
save_png(fig, FIG_DIR / 'V5bp_J_across_width.png')
for name, p in jprof.items():
    print(f"{name}: |i_side / i_port| (tuned) = {p['i_side_over_i_port']:.4f}")

wrote D:\PhD_Fahdilou_Leat\PUBLICATIONS\7-EurAAP Small Antenna Super-Gain Challenge\fiches_cst\resultats_V5\figures\V5bp_J_across_width.png
ns16_nw4: |i_side / i_port| (tuned) = 0.9430
ns16_nw8: |i_side / i_port| (tuned) = 0.9540


## 6. Why is L erratic? Charge neutrality of the Φ term, mirrored vs full-fill Z

For RWG functions, the scalar-potential part is Z_Φ = (1/jωε) Dᵀ K D, where D maps edge currents to the charge density of each
triangle (∇·f_n = ±l_n/A on T±) and K_ts = ∫∫ G over the triangle pair (t, s). For a **solenoidal** current (D u = 0, a loop
current) uᴴ Z_Φ u = 0 **exactly, whatever the quadrature**, provided every entry uses the same K_ts. This matters a lot here:
Z_Φ is ~1/(kh)² ≈ 10⁶ times larger than Z_A, so even a 10⁻⁴ violation of that cancellation (the measured quadrature asymmetry
of EDGE/VERTEX pairs, K_ts ≠ K_st) swamps the inductive part of a loop.

`impedance_matrix_z` fills m ≤ n only and mirrors: entries with m > n then use K_st instead of K_ts, which breaks the Dᵀ K D
structure. With `full_fill=True` every entry uses K_ts consistently. Test (CPU, small meshes): Rayleigh ratio uᴴ Z_Φ u / uᴴ Z_A u
for random exactly-solenoidal u (null space of D), then L = X/ω recomputed with a full-fill Z (and with its symmetric part
(Z + Zᵀ)/2 = Dᵀ K_sym D + …, which also keeps the structure). Z_Φ alone: μ = 0 (pref_A = 0); Z_A alone: ε → 10²⁵⁰.

In [8]:
from scipy.linalg import null_space


def charge_matrix(mesh):
    """D (N_tri, N): D[t, n] = +l_n/A_t if t = T+_n, -l_n/A_t if t = T-_n  (divergence of f_n on t)."""
    tp, tm = mesh.triangles.triangles_plus, mesh.triangles.triangles_minus
    L, A = mesh.edges.edges_length, mesh.triangles.triangles_area
    n = np.arange(mesh.n_edges)
    D = np.zeros((mesh.triangles.total_of_triangles, mesh.n_edges))
    D[tp, n] += L / A[tp]
    D[tm, n] -= L / A[tm]
    return D


def max_aspect(mesh):
    """Largest triangle aspect ratio (longest edge^2 / (2 area))."""
    tri, p = mesh.triangles.triangles[:3], mesh.points.points
    e = [np.linalg.norm(p[:, tri[i]] - p[:, tri[(i + 1) % 3]], axis=0) for i in range(3)]
    return float(np.max(np.max(e, axis=0) ** 2 / (2 * mesh.triangles.triangles_area)))


def k_omega(f):
    w = 2 * np.pi * f
    return 1j * w / C_LIGHT, w


def L_from_Z(mesh, Zf):
    A1 = incidence_matrix([mesh.port('feed')], mesh.edges)
    return {f: (1 / ReducedMultiport.from_matrices(Z, A1, f).Y[0, 0]).imag / (2 * np.pi * f) * 1e9 for f, Z in Zf.items()}


breakdown = {}
rng = np.random.default_rng(0)
for name in ('ns16_nw2', 'ns8_nw4'):
    mesh = meshes[name]
    t0 = time.perf_counter()
    kc, w = k_omega(F0)
    Z_A = impedance_matrix_z(mesh.edges, mesh.triangles, mesh.points, kc, w, MU0, EPS_HUGE)
    Z_P_mir = impedance_matrix_z(mesh.edges, mesh.triangles, mesh.points, kc, w, 0.0, EPS0)
    Z_P_full = impedance_matrix_z(mesh.edges, mesh.triangles, mesh.points, kc, w, 0.0, EPS0, full_fill=True)
    S = null_space(charge_matrix(mesh))                     # exactly solenoidal (charge-free) currents
    ratios = {}
    for label, ZP in (('mirrored (default)', Z_P_mir), ('full fill', Z_P_full), ('full fill, symmetrised', 0.5 * (Z_P_full + Z_P_full.T))):
        r = []
        for _ in range(5):
            u = S @ rng.standard_normal(S.shape[1])
            r.append(abs(u @ ZP @ u) / abs(u @ Z_A @ u))
        ratios[label] = float(np.max(r))
    # L with a full-fill Z at both frequencies
    Zfull = {}
    for f in (F_LOW, F0):
        kc_f, w_f = k_omega(f)
        Zfull[f] = impedance_matrix_z(mesh.edges, mesh.triangles, mesh.points, kc_f, w_f, MU0, EPS0, full_fill=True)
    L_mir = L_from_Z(mesh, Zs[name])
    L_full = L_from_Z(mesh, Zfull)
    L_sym = L_from_Z(mesh, {f: 0.5 * (Z + Z.T) for f, Z in Zfull.items()})
    breakdown[name] = dict(N=mesh.n_edges, solenoidal_dim=S.shape[1], max_aspect=max_aspect(mesh),
                           phi_over_A_on_solenoidal=ratios,
                           L_mirrored_nH={str(f): v for f, v in L_mir.items()},
                           L_full_fill_nH={str(f): v for f, v in L_full.items()},
                           L_full_fill_sym_nH={str(f): v for f, v in L_sym.items()},
                           t_cpu_s=time.perf_counter() - t0)
    print(f"{name} (N = {mesh.n_edges}, max aspect ratio {breakdown[name]['max_aspect']:.1f}, "
          f"{S.shape[1]} solenoidal directions, CPU {breakdown[name]['t_cpu_s']:.0f} s)")
    for label, v in ratios.items():
        print(f"   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, {label:24s}: {v:.3e}")
    print(f"   L = X/w (nH)   mirrored: {L_mir[F_LOW]:8.2f} @0.2 GHz {L_mir[F0]:8.2f} @0.889 GHz | "
          f"full fill: {L_full[F_LOW]:7.2f} / {L_full[F0]:7.2f} | full fill sym.: {L_sym[F_LOW]:7.2f} / {L_sym[F0]:7.2f}")
print(f"references: Grover {L_GROVER*1e9:.1f} nH (static), CST X/w at 0.9 GHz {L_CST_UNTUNED*1e9:.1f} nH")

ns16_nw2 (N = 360, max aspect ratio 2.0, 73 solenoidal directions, CPU 133 s)
   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, mirrored (default)      : 1.074e-01
   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, full fill               : 5.064e-13
   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, full fill, symmetrised  : 5.255e-13
   L = X/w (nH)   mirrored:    83.85 @0.2 GHz    67.18 @0.889 GHz | full fill:   48.82 /   63.86 | full fill sym.:   48.82 /   63.86


ns8_nw4 (N = 528, max aspect ratio 7.8, 145 solenoidal directions, CPU 311 s)
   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, mirrored (default)      : 1.026e-01
   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, full fill               : 4.981e-13
   max |u^H Z_Phi u| / |u^H Z_A u| over solenoidal u, full fill, symmetrised  : 4.256e-13
   L = X/w (nH)   mirrored:    34.96 @0.2 GHz   -50.22 @0.889 GHz | full fill:   47.53 /   61.36 | full fill sym.:   47.53 /   61.36
references: Grover 46.7 nH (static), CST X/w at 0.9 GHz 48.2 nH


In [9]:
summary = dict(frequency_GHz=F0 / 1e9, low_frequency_GHz=F_LOW / 1e9, side_m=SIDE, width_m=W, grading='cosine',
               references=dict(L_grover_nH=L_GROVER * 1e9, L_cst_untuned_nH=L_CST_UNTUNED * 1e9, R_small_loop_ohm=R_SMALL_LOOP),
               delta_and_ding={k: {kk: (vv if not isinstance(vv, complex) else [vv.real, vv.imag])
                                   for kk, vv in v.items() if kk != 'I_tuned'} for k, v in res.items()},
               magnetic_energy=wm, phi_cancellation_and_full_fill=breakdown, J_profiles=jprof)
out = FIG_DIR / 'V5bp_reactance_diagnostic.json'
out.write_text(json.dumps(summary, indent=2, default=float), encoding='utf-8')
print('wrote', out)

wrote D:\PhD_Fahdilou_Leat\PUBLICATIONS\7-EurAAP Small Antenna Super-Gain Challenge\fiches_cst\resultats_V5\figures\V5bp_reactance_diagnostic.json
